# In-Depth Analysis of Homomorphic Encryption Libraries
**Ben-Gurion University — Confidential Computing Course**

This notebook compares **TenSEAL** and **OpenFHE** (both using the CKKS scheme) against a plaintext baseline.

**Operations benchmarked:** key generation, encrypt, add, multiply, sum, average, dot product, decrypt
**Metrics:** runtime (mean over many repetitions), peak Python-side memory, approximation error, ciphertext size

### Measurement methodology

Every timed figure is a **mean over `REPEATS` repetitions**, taken after a **warm-up phase that is discarded**.

The warm-up matters: the first calls into either library pay one-off costs that are not part of the steady-state cost of an operation — loading the native library, lazily building NTT tables and evaluation-key caches, first-touch page faults, and CPU frequency ramp-up. Including them inflates the mean and makes it depend on how many repetitions were run.

Timing and memory are measured in **separate passes**, because `tracemalloc` hooks every Python allocation and measurably distorts short timings. Garbage collection is disabled inside the timed loop, as `timeit` does.

### Applied use case

Section 9 evaluates a **synthetic medical risk score** over a synthetic cohort of 1000 patients — a computation with genuine multiplicative depth (interaction terms and a quadratic term), not just a weighted average.

> **The risk score is synthetic and for demonstration only. It is not a validated clinical model and carries no medical meaning.** The cohort is synthetic too — values are drawn from plausible-looking distributions, not from patients.

## 0. Setup

**OpenFHE version constraint:** the OpenFHE PyPI wheels are Linux-only *and* are built for a specific CPython version — the current wheel ships a `cpython-38` shared object, so it imports only under **Python 3.8**. On a newer runtime the install appears to succeed and then fails at import with `No module named 'openfhe.openfhe'`.

The notebook handles this: if OpenFHE cannot be imported, every section runs with the plaintext and TenSEAL results only.

In [ ]:
!pip install tenseal openfhe matplotlib numpy -q

In [ ]:
import gc
import statistics
import sys
import time
import tracemalloc
import warnings

warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import tenseal as ts

try:
    from openfhe import CCParamsCKKSRNS, GenCryptoContext, PKESchemeFeature
    OPENFHE_AVAILABLE = True
    OPENFHE_ERROR = None
except (ImportError, ModuleNotFoundError) as exc:
    OPENFHE_AVAILABLE = False
    OPENFHE_ERROR = str(exc)

print('python  version:', sys.version.split()[0])
print('tenseal version:', ts.__version__)
if OPENFHE_AVAILABLE:
    print('openfhe        : available')
else:
    print('openfhe        : NOT available ->', OPENFHE_ERROR)
    print('                 (needs Python 3.8; this runtime is', sys.version.split()[0] + ')')
    print('                 Plaintext and TenSEAL sections will still run.')

## 1. Measurement harness

`time_operation` runs a discarded warm-up phase, then times `repeats` individual calls and returns the mean together with dispersion statistics. This is the same harness as `benchmark_harness.py` in the repository.

In [ ]:
Z95 = 1.959963984540054  # two-sided 95% normal quantile


def percentile(sorted_values, pct):
    if len(sorted_values) == 1:
        return sorted_values[0]
    rank = (pct / 100.0) * (len(sorted_values) - 1)
    low = int(rank)
    high = min(low + 1, len(sorted_values) - 1)
    return sorted_values[low] + (rank - low) * (sorted_values[high] - sorted_values[low])


def summarize(samples_s, warmup=0):
    samples_ms = sorted(s * 1000.0 for s in samples_s)
    n = len(samples_ms)
    mean_ms = statistics.fmean(samples_ms)
    std_ms = statistics.stdev(samples_ms) if n > 1 else 0.0
    return {
        'mean_ms': mean_ms,
        'std_ms': std_ms,
        'median_ms': statistics.median(samples_ms),
        'min_ms': samples_ms[0],
        'max_ms': samples_ms[-1],
        'p95_ms': percentile(samples_ms, 95.0),
        'ci95_half_width_ms': Z95 * std_ms / (n ** 0.5) if n > 1 else 0.0,
        'rel_std_pct': (std_ms / mean_ms * 100.0) if mean_ms > 0 else 0.0,
        'repeats': n,
        'warmup': warmup,
    }


def time_operation(fn, repeats, warmup):
    """Warm up (discarded), then time `repeats` individual calls."""
    for _ in range(warmup):
        fn()

    samples_s = []
    gc_was_enabled = gc.isenabled()
    gc.disable()          # as timeit does: don't charge an unrelated collection to fn
    try:
        for _ in range(repeats):
            start = time.perf_counter()
            fn()
            samples_s.append(time.perf_counter() - start)
    finally:
        if gc_was_enabled:
            gc.enable()
    return summarize(samples_s, warmup=warmup)


def measure_peak_memory_kb(fn, repeats=5):
    """Separate pass: tracemalloc would distort the timings above."""
    fn()
    peaks = []
    for _ in range(repeats):
        tracemalloc.start()
        fn()
        peaks.append(tracemalloc.get_traced_memory()[1] / 1024.0)
        tracemalloc.stop()
    return statistics.fmean(peaks)


def benchmark_operations(operations, repeats, warmup, memory_repeats=5, label=''):
    results = {}
    for key, fn in operations.items():
        print(f'  [{label}] {key:<12} {repeats} reps (+{warmup} warm-up)...', end='', flush=True)
        stats = time_operation(fn, repeats, warmup)
        results[f'{key}_time_stats'] = stats
        results[f'{key}_time_s'] = stats['mean_ms'] / 1000.0
        results[f'{key}_mem_kb'] = measure_peak_memory_kb(fn, memory_repeats)
        print(f" mean {stats['mean_ms']:9.4f} ms +/- {stats['ci95_half_width_ms']:.4f}"
              f" ({stats['rel_std_pct']:.1f}% rel. sd)", flush=True)
    results['repeats'] = repeats
    results['warmup'] = warmup
    return results

## 2. Plaintext baseline

Each operation is timed separately so plaintext and encrypted figures are comparable per operation, not only in aggregate.

In [ ]:
def plaintext_benchmark(vector, repeats, warmup):
    n = len(vector)
    vector2 = [v * 0.5 + 1.0 for v in vector]
    operations = {
        'add': lambda: [v + v for v in vector],
        'mul': lambda: [v * v for v in vector],
        'sum': lambda: sum(vector),
        'avg': lambda: sum(vector) / n,
        'dot': lambda: sum(a * b for a, b in zip(vector, vector2)),
    }
    results = benchmark_operations(operations, repeats, warmup, label='Plaintext')
    results['total_time_s'] = sum(results[f'{k}_time_s'] for k in ['add', 'mul', 'sum', 'avg', 'dot'])
    return results

## 3. TenSEAL benchmark (CKKS)

In [ ]:
def make_tenseal_context():
    ctx = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60],
    )
    ctx.generate_galois_keys()
    ctx.global_scale = 2 ** 40
    return ctx


def tenseal_benchmark(vector, repeats, warmup):
    ctx = make_tenseal_context()
    n = len(vector)
    vector2 = [v * 0.5 + 1.0 for v in vector]

    # Fresh operands, reused by every repetition, so no repetition inherits
    # accumulated noise or a consumed multiplicative level.
    enc = ts.ckks_vector(ctx, vector)
    enc2 = ts.ckks_vector(ctx, vector2)

    operations = {
        'encrypt': lambda: ts.ckks_vector(ctx, vector),
        'add': lambda: enc + enc,
        'mul': lambda: enc * enc,
        'sum': lambda: enc.sum(),
        'avg': lambda: enc.sum() * (1.0 / n),
        'dot': lambda: (enc * enc2).sum(),
        'decrypt': lambda: enc.decrypt(),
    }
    results = benchmark_operations(operations, repeats, warmup, label='TenSEAL')

    dec_add = (enc + enc).decrypt()
    dec_mul = (enc * enc).decrypt()
    dec_sum = enc.sum().decrypt()[0]
    dec_avg = (enc.sum() * (1.0 / n)).decrypt()[0]
    dec_dot = (enc * enc2).sum().decrypt()[0]

    expected_dot = sum(a * b for a, b in zip(vector, vector2))
    results['add_max_error'] = max(abs(a - (v + v)) for a, v in zip(dec_add, vector))
    results['mul_max_error'] = max(abs(a - (v * v)) for a, v in zip(dec_mul, vector))
    results['sum_error'] = abs(dec_sum - sum(vector))
    results['avg_error'] = abs(dec_avg - sum(vector) / n)
    results['dot_error'] = abs(dec_dot - expected_dot)
    results['ciphertext_bytes'] = len(enc.serialize())
    return results

## 4. OpenFHE benchmark (CKKS)

In [ ]:
def next_power_of_two(n):
    p = 1
    while p < n:
        p <<= 1
    return p


def make_openfhe_context(batch_size, multiplicative_depth=3, scaling_mod_size=50):
    params = CCParamsCKKSRNS()
    params.SetMultiplicativeDepth(multiplicative_depth)
    params.SetScalingModSize(scaling_mod_size)
    params.SetBatchSize(next_power_of_two(batch_size))
    cc = GenCryptoContext(params)
    for feature in (PKESchemeFeature.PKE, PKESchemeFeature.KEYSWITCH,
                    PKESchemeFeature.LEVELEDSHE, PKESchemeFeature.ADVANCEDSHE):
        cc.Enable(feature)
    keys = cc.KeyGen()
    cc.EvalMultKeyGen(keys.secretKey)
    cc.EvalSumKeyGen(keys.secretKey)
    return cc, keys


def openfhe_benchmark(vector, repeats, warmup):
    n = len(vector)
    vector2 = [v * 0.5 + 1.0 for v in vector]
    cc, keys = make_openfhe_context(n)

    ct = cc.Encrypt(keys.publicKey, cc.MakeCKKSPackedPlaintext(vector))
    ct2 = cc.Encrypt(keys.publicKey, cc.MakeCKKSPackedPlaintext(vector2))

    operations = {
        'encrypt': lambda: cc.Encrypt(keys.publicKey, cc.MakeCKKSPackedPlaintext(vector)),
        'add': lambda: cc.EvalAdd(ct, ct),
        'mul': lambda: cc.EvalMult(ct, ct),
        'sum': lambda: cc.EvalSum(ct, n),
        'avg': lambda: cc.EvalMult(cc.EvalSum(ct, n), 1.0 / n),
        'dot': lambda: cc.EvalInnerProduct(ct, ct2, n),
        'decrypt': lambda: cc.Decrypt(keys.secretKey, ct),
    }
    results = benchmark_operations(operations, repeats, warmup, label='OpenFHE')

    def dec(ciphertext, length):
        plain = cc.Decrypt(keys.secretKey, ciphertext)
        plain.SetLength(length)
        return plain.GetRealPackedValue()

    dec_add = dec(cc.EvalAdd(ct, ct), n)
    dec_mul = dec(cc.EvalMult(ct, ct), n)
    dec_sum = dec(cc.EvalSum(ct, n), 1)[0]
    dec_avg = dec(cc.EvalMult(cc.EvalSum(ct, n), 1.0 / n), 1)[0]
    dec_dot = dec(cc.EvalInnerProduct(ct, ct2, n), 1)[0]

    expected_dot = sum(a * b for a, b in zip(vector, vector2))
    results['add_max_error'] = max(abs(a - (v + v)) for a, v in zip(dec_add, vector))
    results['mul_max_error'] = max(abs(a - (v * v)) for a, v in zip(dec_mul, vector))
    results['sum_error'] = abs(dec_sum - sum(vector))
    results['avg_error'] = abs(dec_avg - sum(vector) / n)
    results['dot_error'] = abs(dec_dot - expected_dot)
    results['ring_dimension'] = cc.GetRingDimension()
    return results

## 5. Run all benchmarks

`REPEATS = 1000` reproduces the figures in the report. Lower it for a quick pass — the code path is identical, only the confidence intervals widen.

In [ ]:
DATA = [1.0, 2.0, 3.0, 4.0, 5.0]
REPEATS = 1000
WARMUP = 50
KEYGEN_REPEATS = 50   # one-off setup cost, so fewer repetitions

print(f'Input vector: {DATA}')
print(f'Timing: {REPEATS} repetitions per operation, {WARMUP} discarded warm-up calls\n')

print('Running plaintext baseline...')
pt = plaintext_benchmark(DATA, REPEATS, WARMUP)

print('Running TenSEAL benchmark...')
tenseal_res = tenseal_benchmark(DATA, REPEATS, WARMUP)

if OPENFHE_AVAILABLE:
    print('Running OpenFHE benchmark...')
    openfhe_res = openfhe_benchmark(DATA, REPEATS, WARMUP)
else:
    openfhe_res = None
    print('Skipping OpenFHE (not importable on this runtime).')

print('\nTiming key generation...')
keygen = {'tenseal': time_operation(make_tenseal_context, KEYGEN_REPEATS, 5)}
if OPENFHE_AVAILABLE:
    keygen['openfhe'] = time_operation(lambda: make_openfhe_context(len(DATA)),
                                       KEYGEN_REPEATS, 5)
print('Done.')

## 6. Comparison tables

In [ ]:
OPS = ['Add', 'Multiply', 'Sum', 'Average', 'Dot Product']
OP_KEYS = ['add', 'mul', 'sum', 'avg', 'dot']
ALL_OPS = ['Encrypt', 'Add', 'Multiply', 'Sum', 'Average', 'Dot Product', 'Decrypt']
ALL_OP_KEYS = ['encrypt', 'add', 'mul', 'sum', 'avg', 'dot', 'decrypt']

ts_res = tenseal_res
fhe = openfhe_res
W = 92


def mean_ms(results, key):
    stats = results.get(f'{key}_time_stats') if results else None
    return stats['mean_ms'] if stats else None


print('=' * W)
print(f"{'Mean Time per Operation (ms)':^{W}}")
print('=' * W)
print(f"{'Operation':<14}{'Plaintext':>14}{'TenSEAL':>14}{'OpenFHE':>16}"
      f"{'TS overhead':>16}{'FHE overhead':>16}")
print('-' * W)
for name, key in zip(ALL_OPS, ALL_OP_KEYS):
    p, t = mean_ms(pt, key), mean_ms(ts_res, key)
    f = mean_ms(fhe, key)
    p_s = f'{p:>14.6f}' if p is not None else f"{'-':>14}"
    f_s = f'{f:>16.4f}' if f is not None else f"{'n/a':>16}"
    t_o = f'{t / p:>15,.0f}x' if (p and t) else f"{'-':>16}"
    f_o = f'{f / p:>15,.0f}x' if (p and f) else f"{'-':>16}"
    print(f'{name:<14}{p_s}{t:>14.4f}{f_s}{t_o}{f_o}')
print('-' * W)

pt_total = sum(mean_ms(pt, k) for k in OP_KEYS)
ts_total = sum(mean_ms(ts_res, k) for k in ALL_OP_KEYS)
print(f"{'Total':<14}{pt_total:>14.6f}{ts_total:>14.4f}", end='')
if fhe:
    fhe_total = sum(mean_ms(fhe, k) for k in ALL_OP_KEYS)
    print(f'{fhe_total:>16.4f}{ts_total / pt_total:>15,.0f}x{fhe_total / pt_total:>15,.0f}x')
else:
    print(f"{'n/a':>16}{ts_total / pt_total:>15,.0f}x{'-':>16}")

print()
print('=' * W)
print(f"{'Measurement Stability — this is what the repetitions buy':^{W}}")
print('=' * W)
print(f"{'Operation':<14}{'TenSEAL mean':>15}{'rel. SD':>11}{'95% CI +/-':>13}"
      f"{'OpenFHE mean':>15}{'rel. SD':>11}{'95% CI +/-':>13}")
print('-' * W)
for name, key in zip(ALL_OPS, ALL_OP_KEYS):
    st = ts_res[f'{key}_time_stats']
    row = f"{name:<14}{st['mean_ms']:>15.4f}{st['rel_std_pct']:>10.1f}%{st['ci95_half_width_ms']:>13.4f}"
    if fhe:
        fs = fhe[f'{key}_time_stats']
        row += f"{fs['mean_ms']:>15.4f}{fs['rel_std_pct']:>10.1f}%{fs['ci95_half_width_ms']:>13.4f}"
    print(row)
print('-' * W)

print()
print('=' * W)
print(f"{'Context + Key Generation (one-off setup cost)':^{W}}")
print('=' * W)
print(f"{'Library':<16}{'Mean (ms)':>14}{'SD (ms)':>12}{'95% CI +/-':>13}")
print('-' * W)
for label, key in [('TenSEAL', 'tenseal'), ('OpenFHE', 'openfhe')]:
    st = keygen.get(key)
    if st:
        print(f"{label:<16}{st['mean_ms']:>14.3f}{st['std_ms']:>12.3f}{st['ci95_half_width_ms']:>13.3f}")
print('-' * W)

print()
print('=' * W)
print(f"{'Approximation Error (max absolute error)':^{W}}")
print('=' * W)
print(f"{'Operation':<16}{'TenSEAL':>22}{'OpenFHE':>22}")
print('-' * W)
for name, key in zip(OPS, ['add_max_error', 'mul_max_error', 'sum_error', 'avg_error', 'dot_error']):
    f_s = f'{fhe[key]:>22.4e}' if fhe else f"{'n/a':>22}"
    print(f'{name:<16}{ts_res[key]:>22.4e}{f_s}')
print('-' * W)
print(f"\nTenSEAL ciphertext size: {ts_res['ciphertext_bytes']:,} bytes"
      f' (plaintext vector: {len(DATA) * 8} bytes as float64)')
if fhe:
    print(f"OpenFHE ring dimension : {fhe['ring_dimension']}")

## 7. Charts

Bars carry 95% confidence intervals on the mean. Series colours are a colourblind-safe categorical set.

In [ ]:
C_TS, C_FHE, C_PT = '#2a78d6', '#eb6834', '#1baf7a'
INK, INK_SOFT, GRID = '#0b0b0b', '#52514e', '#dedddA'

plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': GRID, 'grid.color': GRID, 'grid.linestyle': '-',
    'grid.linewidth': 0.6, 'legend.frameon': False,
    'xtick.color': INK_SOFT, 'ytick.color': INK_SOFT,
})

BAR_W, GAP = 0.38, 0.02


def stats_of(results, keys):
    means = [results[f'{k}_time_stats']['mean_ms'] for k in keys]
    errs = [results[f'{k}_time_stats']['ci95_half_width_ms'] for k in keys]
    return np.array(means), np.array(errs)


def label_bars(ax, bars, values, fmt='{:.2f}'):
    for bar, value in zip(bars, values):
        ax.annotate(fmt.format(value), xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom',
                    fontsize=7.5, color=INK_SOFT)


x = np.arange(len(ALL_OPS))
ts_mean, ts_err = stats_of(ts_res, ALL_OP_KEYS)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
off = (BAR_W + GAP) / 2 if fhe else 0
b1 = ax.bar(x - off, ts_mean, BAR_W if fhe else 0.55, yerr=ts_err, capsize=2.5,
            label='TenSEAL', color=C_TS, error_kw=dict(ecolor=INK_SOFT, lw=0.8))
label_bars(ax, b1, ts_mean)
if fhe:
    fhe_mean, fhe_err = stats_of(fhe, ALL_OP_KEYS)
    b2 = ax.bar(x + off, fhe_mean, BAR_W, yerr=fhe_err, capsize=2.5, label='OpenFHE',
                color=C_FHE, error_kw=dict(ecolor=INK_SOFT, lw=0.8))
    label_bars(ax, b2, fhe_mean)
ax.set_title('Mean operation time (ms)')
ax.set_xticks(x); ax.set_xticklabels(ALL_OPS, rotation=20, ha='right')
ax.set_ylabel('Time (ms) - log scale'); ax.set_yscale('log')
ax.grid(True, axis='y', alpha=0.6); ax.set_axisbelow(True); ax.legend(loc='upper left')

ax = axes[1]
ts_mem = [ts_res[f'{k}_mem_kb'] for k in OP_KEYS]
xm = np.arange(len(OPS))
b3 = ax.bar(xm - off, ts_mem, BAR_W if fhe else 0.55, label='TenSEAL', color=C_TS)
label_bars(ax, b3, ts_mem)
if fhe:
    fhe_mem = [fhe[f'{k}_mem_kb'] for k in OP_KEYS]
    b4 = ax.bar(xm + off, fhe_mem, BAR_W, label='OpenFHE', color=C_FHE)
    label_bars(ax, b4, fhe_mem)
ax.set_title('Peak Python-side memory (KB)')
ax.set_xticks(xm); ax.set_xticklabels(OPS, rotation=20, ha='right')
ax.set_ylabel('KB'); ax.grid(True, axis='y', alpha=0.6); ax.set_axisbelow(True)
ax.legend()

ax = axes[2]
ERR_KEYS = ['add_max_error', 'mul_max_error', 'sum_error', 'avg_error', 'dot_error']
ts_errs = [ts_res[k] for k in ERR_KEYS]
b5 = ax.bar(xm - off, ts_errs, BAR_W if fhe else 0.55, label='TenSEAL', color=C_TS)
label_bars(ax, b5, ts_errs, fmt='{:.1e}')
if fhe:
    fhe_errs = [fhe[k] for k in ERR_KEYS]
    b6 = ax.bar(xm + off, fhe_errs, BAR_W, label='OpenFHE', color=C_FHE)
    label_bars(ax, b6, fhe_errs, fmt='{:.1e}')
ax.set_title('Approximation error (log)')
ax.set_xticks(xm); ax.set_xticklabels(OPS, rotation=20, ha='right')
ax.set_ylabel('Max abs error'); ax.set_yscale('log')
ax.grid(True, axis='y', alpha=0.6); ax.set_axisbelow(True); ax.legend()

plt.suptitle('TenSEAL vs OpenFHE - CKKS benchmark', fontsize=14, fontweight='bold')
plt.figtext(0.5, 0.005, f'Mean of {REPEATS:,} timed repetitions per operation '
                        f'({WARMUP} warm-up calls discarded); bars show 95% CI.',
            ha='center', fontsize=8, color=INK_SOFT)
plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig('chart_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Scaling benchmark (different vector sizes)

Context and key generation happen once per size and are excluded from the timed pipeline.

In [ ]:
SIZES = [5, 10, 50, 100, 500]
SCALING_REPEATS = 200   # a full pipeline per repetition; raise to 1000 to match the report
SCALING_WARMUP = 20


def tenseal_pipeline(vector):
    ctx = make_tenseal_context()
    n = len(vector)

    def pipeline():
        enc = ts.ckks_vector(ctx, vector)
        (enc + enc).decrypt()
        (enc * enc).decrypt()
        enc.sum().decrypt()
        (enc.sum() * (1.0 / n)).decrypt()
    return pipeline


def openfhe_pipeline(vector):
    n = len(vector)
    cc, keys = make_openfhe_context(n)

    def dec(ciphertext, length):
        plain = cc.Decrypt(keys.secretKey, ciphertext)
        plain.SetLength(length)
        return plain.GetRealPackedValue()

    def pipeline():
        ct = cc.Encrypt(keys.publicKey, cc.MakeCKKSPackedPlaintext(vector))
        dec(cc.EvalAdd(ct, ct), n)
        dec(cc.EvalMult(ct, ct), n)
        dec(cc.EvalSum(ct, n), 1)
        dec(cc.EvalMult(cc.EvalSum(ct, n), 1.0 / n), 1)
    return pipeline


def plaintext_pipeline(vector):
    n = len(vector)

    def pipeline():
        [v + v for v in vector]
        [v * v for v in vector]
        sum(vector)
        sum(vector) / n
    return pipeline


pt_scaling, ts_scaling, fhe_scaling = [], [], []
pt_ci, ts_ci, fhe_ci = [], [], []

for size in SIZES:
    vec = [float(i + 1) for i in range(size)]
    print(f'size={size}...', end=' ', flush=True)

    s = time_operation(plaintext_pipeline(vec), SCALING_REPEATS, SCALING_WARMUP)
    pt_scaling.append(s['mean_ms']); pt_ci.append(s['ci95_half_width_ms'])
    s = time_operation(tenseal_pipeline(vec), SCALING_REPEATS, SCALING_WARMUP)
    ts_scaling.append(s['mean_ms']); ts_ci.append(s['ci95_half_width_ms'])
    if OPENFHE_AVAILABLE:
        s = time_operation(openfhe_pipeline(vec), SCALING_REPEATS, SCALING_WARMUP)
        fhe_scaling.append(s['mean_ms']); fhe_ci.append(s['ci95_half_width_ms'])

    msg = f'pt={pt_scaling[-1]:.4f}ms  ts={ts_scaling[-1]:.1f}ms'
    if OPENFHE_AVAILABLE:
        msg += f'  fhe={fhe_scaling[-1]:.1f}ms'
    print(msg)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

ax = axes[0]
ax.errorbar(SIZES, ts_scaling, yerr=ts_ci, fmt='o-', color=C_TS, label='TenSEAL',
            linewidth=2, markersize=7, capsize=3, ecolor=INK_SOFT, elinewidth=0.8)
if OPENFHE_AVAILABLE:
    ax.errorbar(SIZES, fhe_scaling, yerr=fhe_ci, fmt='s-', color=C_FHE, label='OpenFHE',
                linewidth=2, markersize=7, capsize=3, ecolor=INK_SOFT, elinewidth=0.8)
ax.set_xlabel('Vector size (elements)'); ax.set_ylabel('Mean pipeline time (ms)')
ax.set_title('Encrypted pipeline scaling'); ax.grid(True, alpha=0.6)
ax.set_axisbelow(True); ax.legend()

ax = axes[1]
ax.errorbar(SIZES, pt_scaling, yerr=pt_ci, fmt='^-', color=C_PT, label='Plaintext',
            linewidth=2, markersize=7, capsize=3, ecolor=INK_SOFT, elinewidth=0.8)
ax.errorbar(SIZES, ts_scaling, yerr=ts_ci, fmt='o-', color=C_TS, label='TenSEAL',
            linewidth=2, markersize=7, capsize=3, ecolor=INK_SOFT, elinewidth=0.8)
if OPENFHE_AVAILABLE:
    ax.errorbar(SIZES, fhe_scaling, yerr=fhe_ci, fmt='s-', color=C_FHE, label='OpenFHE',
                linewidth=2, markersize=7, capsize=3, ecolor=INK_SOFT, elinewidth=0.8)
ax.set_xlabel('Vector size (elements)')
ax.set_ylabel('Mean pipeline time (ms) - log scale')
ax.set_title('Plaintext vs encrypted'); ax.set_yscale('log')
ax.grid(True, which='major', alpha=0.6); ax.set_axisbelow(True); ax.legend()

plt.suptitle('Performance scaling with vector size', fontsize=13, fontweight='bold')
plt.tight_layout(rect=(0, 0.02, 1, 0.95))
plt.savefig('chart_scaling_full.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Applied use case — encrypted synthetic medical risk score

> **Disclaimer.** The risk score below is a **synthetic construct for demonstration purposes only**. It is **not** a validated clinical model, it is not derived from medical literature, and it must not be interpreted as a real assessment of patient risk. The cohort is synthetic as well — values are drawn from plausible-looking distributions, not from patients.

**Scenario.** A hospital holds records for 1000 patients with five measures each. It wants a cloud provider to compute a risk score per patient *and* the cohort mean score, without ever exposing an individual value. The hospital encrypts one ciphertext per feature; the cloud evaluates the whole score on ciphertexts; only the hospital can decrypt.

**Why this is a better HE benchmark than a single average.** Each feature is first min-max normalized against a *public* reference range:

$$x_{norm} = \frac{x - min}{max - min}$$

The score then combines the normalized features with a weighted sum **plus two interaction terms and one quadratic term**:

$$\text{risk} = 100 \cdot \big(0.15\,age + 0.20\,bp + 0.15\,bmi + 0.15\,chol + 0.15\,gluc + 0.10\,bp \cdot gluc + 0.05\,bmi \cdot chol + 0.05\,bp^2\big)$$

The last three terms require **ciphertext x ciphertext** multiplication, not just multiplication by a public constant, so the circuit consumes real multiplicative depth. The eight weights sum to 1.0, so a patient at the top of every reference range scores 100.

**Packing.** One feature per ciphertext, one CKKS slot per patient, so a single homomorphic multiplication advances all 1000 patients at once. Vectors are padded to a power of two (OpenFHE requires it); padding slots are filled with each feature's range *minimum* so they normalize to exactly 0 and drop out of the cohort sum — masking them would otherwise cost an extra level.

**Why the reference ranges are public.** Deriving min/max from the data would need comparisons between ciphertexts, which CKKS does not support natively. Public ranges keep normalization an affine map, which CKKS does cheaply.

In [ ]:
FEATURES = ('age', 'systolic_bp', 'bmi', 'cholesterol', 'glucose')

RANGES = {
    'age': (18.0, 90.0),
    'systolic_bp': (90.0, 180.0),
    'bmi': (18.0, 40.0),
    'cholesterol': (120.0, 280.0),
    'glucose': (70.0, 200.0),
}
DISTRIBUTIONS = {
    'age': {'kind': 'uniform'},
    'systolic_bp': {'kind': 'normal', 'mean': 125.0, 'sd': 15.0},
    'bmi': {'kind': 'normal', 'mean': 27.0, 'sd': 4.5},
    'cholesterol': {'kind': 'normal', 'mean': 200.0, 'sd': 35.0},
    'glucose': {'kind': 'normal', 'mean': 100.0, 'sd': 20.0},
}
LINEAR_WEIGHTS = {'age': 0.15, 'systolic_bp': 0.20, 'bmi': 0.15,
                  'cholesterol': 0.15, 'glucose': 0.15}
INTERACTION_WEIGHTS = {('systolic_bp', 'glucose'): 0.10, ('bmi', 'cholesterol'): 0.05}
SQUARE_WEIGHTS = {'systolic_bp': 0.05}
SCORE_SCALE = 100.0

N_PATIENTS = 1000
SEED = 20260730


def generate_cohort(n_patients=N_PATIENTS, seed=SEED):
    rng = np.random.default_rng(seed)
    cohort = {}
    for name in FEATURES:
        low, high = RANGES[name]
        spec = DISTRIBUTIONS[name]
        if spec['kind'] == 'uniform':
            values = rng.uniform(low, high, size=n_patients)
        else:
            values = rng.normal(spec['mean'], spec['sd'], size=n_patients)
        cohort[name] = np.clip(values, low, high)   # keep every x_norm inside [0, 1]
    return cohort


def normalization_affine_terms(name):
    """(scale, offset) with x_norm == x * scale + offset."""
    low, high = RANGES[name]
    return 1.0 / (high - low), -low / (high - low)


def plaintext_risk_scores(cohort):
    norm = {n: (np.asarray(cohort[n], float) - RANGES[n][0]) / (RANGES[n][1] - RANGES[n][0])
            for n in FEATURES}
    total = np.zeros_like(norm[FEATURES[0]])
    for name, weight in LINEAR_WEIGHTS.items():
        total = total + weight * norm[name]
    for (a, b), weight in INTERACTION_WEIGHTS.items():
        total = total + weight * norm[a] * norm[b]
    for name, weight in SQUARE_WEIGHTS.items():
        total = total + weight * norm[name] * norm[name]
    return SCORE_SCALE * total


cohort = generate_cohort()
reference_scores = plaintext_risk_scores(cohort)

print('!! SYNTHETIC score, demonstration only - not a validated clinical model.\n')
print(f'Synthetic cohort: {N_PATIENTS} patients\n')
print(f"{'Feature':<14}{'Public range':>18}{'Mean':>10}{'SD':>10}{'Min':>10}{'Max':>10}")
for name in FEATURES:
    v = np.asarray(cohort[name], float)
    low, high = RANGES[name]
    print(f"{name:<14}{f'[{low:g}, {high:g}]':>18}{v.mean():>10.2f}{v.std():>10.2f}"
          f'{v.min():>10.2f}{v.max():>10.2f}')
print(f'\nRisk score  mean {reference_scores.mean():.4f}  sd {reference_scores.std():.4f}  '
      f'min {reference_scores.min():.4f}  max {reference_scores.max():.4f}')

### 9.1 Crypto parameters for the deeper circuit

The parameters used for the primitive operations **cannot run this circuit**. TenSEAL's `[60, 40, 40, 60]` chain allows two rescalings; the score needs four. We verified this by sweeping: three usable levels fails with `scale out of bounds`, four works. Lengthening the chain pushes the total modulus bit-count past what `poly_modulus_degree = 8192` permits at a 128-bit security level — that context is rejected outright — so the ring dimension has to grow to 16384, which roughly doubles the cost of *every* operation.

The same sweep on OpenFHE: depth 3 evaluates the score but **fails on the cohort mean**, depth 4 completes at ring dimension 16384, and depth 5 also works but pushes the ring to 32768 — twice the work for nothing. Depth 4 is therefore the deliberate choice, and asking for more depth than a circuit needs is a real and easy performance mistake.

This is the central practical lesson of the use case: **circuit depth, not data volume, drives HE cost.** Going from an average to a score with interaction terms costs more than going from 5 patients to 1000.

In [ ]:
def pad_features(cohort, n_slots):
    """Pad each column with its range minimum -> normalizes to 0, drops out of the sum."""
    padded = {}
    for name in FEATURES:
        values = np.asarray(cohort[name], float)
        pad_len = n_slots - len(values)
        padded[name] = (np.concatenate([values, np.full(pad_len, RANGES[name][0])])
                        if pad_len > 0 else values)
    return padded


def scaled_weights():
    """Fold the factor of 100 into each weight, saving one multiplicative level."""
    return ({k: SCORE_SCALE * w for k, w in LINEAR_WEIGHTS.items()},
            {k: SCORE_SCALE * w for k, w in INTERACTION_WEIGHTS.items()},
            {k: SCORE_SCALE * w for k, w in SQUARE_WEIGHTS.items()})


class TenSEALRiskScore:
    name = 'TenSEAL'

    def __init__(self, cohort, n_patients):
        self.n_patients = n_patients
        self.n_slots = next_power_of_two(n_patients)
        self.padded = pad_features(cohort, self.n_slots)
        self.ctx = ts.context(
            ts.SCHEME_TYPE.CKKS,
            poly_modulus_degree=16384,                          # up from 8192
            coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60],       # 4 usable levels
        )
        self.ctx.generate_galois_keys()
        self.ctx.global_scale = 2 ** 40

    def encrypt(self):
        return {n: ts.ckks_vector(self.ctx, self.padded[n].tolist()) for n in FEATURES}

    def score(self, enc):
        linear_w, inter_w, square_w = scaled_weights()
        norm = {}
        for name in FEATURES:
            scale, offset = normalization_affine_terms(name)
            norm[name] = enc[name] * scale + offset
        total = None
        for name, weight in linear_w.items():
            term = norm[name] * weight
            total = term if total is None else total + term
        for (a, b), weight in inter_w.items():
            total = total + (norm[a] * norm[b]) * weight
        for name, weight in square_w.items():
            total = total + (norm[name] * norm[name]) * weight
        return total

    def cohort_mean(self, ct):
        return ct.sum() * (1.0 / self.n_patients)

    def decrypt_scores(self, ct):
        return np.asarray(ct.decrypt()[:self.n_patients])

    def decrypt_mean(self, ct):
        return ct.decrypt()[0]

    def params(self):
        return 'poly_modulus_degree=16384, coeff_mod=[60,40,40,40,40,60], scale=2^40'


class OpenFHERiskScore:
    name = 'OpenFHE'

    def __init__(self, cohort, n_patients):
        self.n_patients = n_patients
        self.n_slots = next_power_of_two(n_patients)
        self.padded = pad_features(cohort, self.n_slots)
        self.cc, self.keys = make_openfhe_context(self.n_slots, multiplicative_depth=4)

    def encrypt(self):
        return {n: self.cc.Encrypt(self.keys.publicKey,
                                   self.cc.MakeCKKSPackedPlaintext(self.padded[n].tolist()))
                for n in FEATURES}

    def score(self, enc):
        linear_w, inter_w, square_w = scaled_weights()
        cc = self.cc
        norm = {}
        for name in FEATURES:
            scale, offset = normalization_affine_terms(name)
            norm[name] = cc.EvalAdd(cc.EvalMult(enc[name], scale), offset)
        total = None
        for name, weight in linear_w.items():
            term = cc.EvalMult(norm[name], weight)
            total = term if total is None else cc.EvalAdd(total, term)
        for (a, b), weight in inter_w.items():
            total = cc.EvalAdd(total, cc.EvalMult(cc.EvalMult(norm[a], norm[b]), weight))
        for name, weight in square_w.items():
            total = cc.EvalAdd(total, cc.EvalMult(cc.EvalMult(norm[name], norm[name]), weight))
        return total

    def cohort_mean(self, ct):
        return self.cc.EvalMult(self.cc.EvalSum(ct, self.n_slots), 1.0 / self.n_patients)

    def _dec(self, ct, length):
        plain = self.cc.Decrypt(self.keys.secretKey, ct)
        plain.SetLength(length)
        return plain.GetRealPackedValue()

    def decrypt_scores(self, ct):
        return np.asarray(self._dec(ct, self.n_patients))

    def decrypt_mean(self, ct):
        return self._dec(ct, 1)[0]

    def params(self):
        return f'multiplicative_depth=4, scaling_mod_size=50, ring_dim={self.cc.GetRingDimension()}'

In [ ]:
RISK_REPEATS = 30    # a full pipeline per repetition; the report uses 1000
RISK_WARMUP = 3

STAGES = ['encrypt_features', 'score_evaluation', 'cohort_mean', 'decrypt_scores']

backends = [TenSEALRiskScore(cohort, N_PATIENTS)]
if OPENFHE_AVAILABLE:
    backends.append(OpenFHERiskScore(cohort, N_PATIENTS))

risk_results = {}
for backend in backends:
    print(f'\n[{backend.name}] {backend.params()}')
    enc = backend.encrypt()
    ct_score = backend.score(enc)
    ct_mean = backend.cohort_mean(ct_score)

    stages = {
        'encrypt_features': lambda b=backend: b.encrypt(),
        'score_evaluation': lambda b=backend, e=enc: b.score(e),
        'cohort_mean': lambda b=backend, c=ct_score: b.cohort_mean(c),
        'decrypt_scores': lambda b=backend, c=ct_score: b.decrypt_scores(c),
    }
    res = benchmark_operations(stages, RISK_REPEATS, RISK_WARMUP, memory_repeats=2,
                              label=backend.name)

    decrypted = backend.decrypt_scores(ct_score)
    errors = np.abs(decrypted - reference_scores)
    res['score_max_abs_error'] = float(errors.max())
    res['score_mean_abs_error'] = float(errors.mean())
    res['cohort_mean_encrypted'] = float(backend.decrypt_mean(ct_mean))
    res['cohort_mean_plaintext'] = float(reference_scores.mean())
    res['cohort_mean_abs_error'] = abs(res['cohort_mean_encrypted'] - res['cohort_mean_plaintext'])
    res['pipeline_total_ms'] = sum(res[f'{s}_time_stats']['mean_ms'] for s in STAGES)
    risk_results[backend.name] = res

pt_score_stats = time_operation(lambda: plaintext_risk_scores(cohort), RISK_REPEATS, RISK_WARMUP)

print()
print('=' * 88)
print(f"{'Risk Score Pipeline - mean time per stage (ms)':^88}")
print('=' * 88)
names = list(risk_results)
print(f"{'Stage':<20}{'Plaintext':>14}" + ''.join(f'{n:>16}' for n in names))
print('-' * 88)
for stage in STAGES:
    row = f'{stage:<20}'
    row += f"{pt_score_stats['mean_ms']:>14.4f}" if stage == 'score_evaluation' else f"{'-':>14}"
    for n in names:
        row += f"{risk_results[n][f'{stage}_time_stats']['mean_ms']:>16.3f}"
    print(row)
print('-' * 88)
row = f"{'Pipeline total':<20}{pt_score_stats['mean_ms']:>14.4f}"
for n in names:
    row += f"{risk_results[n]['pipeline_total_ms']:>16.3f}"
print(row)
row = f"{'Overhead vs plain':<20}{'1x':>14}"
for n in names:
    row += f"{risk_results[n]['pipeline_total_ms'] / pt_score_stats['mean_ms']:>15,.0f}x"
print(row)
row = f"{'Per patient (ms)':<20}{pt_score_stats['mean_ms'] / N_PATIENTS:>14.6f}"
for n in names:
    row += f"{risk_results[n]['pipeline_total_ms'] / N_PATIENTS:>16.4f}"
print(row)

print()
print('=' * 88)
print(f"{'Risk Score Accuracy vs Plaintext Reference':^88}")
print('=' * 88)
print(f"{'Metric':<34}" + ''.join(f'{n:>18}' for n in names))
print('-' * 88)
for label, key, fmt in [
    ('Max abs error (score points)', 'score_max_abs_error', '{:>18.4e}'),
    ('Mean abs error (score points)', 'score_mean_abs_error', '{:>18.4e}'),
    ('Cohort mean, encrypted', 'cohort_mean_encrypted', '{:>18.6f}'),
    ('Cohort mean, plaintext', 'cohort_mean_plaintext', '{:>18.6f}'),
    ('Cohort mean abs error', 'cohort_mean_abs_error', '{:>18.4e}'),
]:
    print(f'{label:<34}' + ''.join(fmt.format(risk_results[n][key]) for n in names))
print('-' * 88)
print('\nThe cloud never saw a raw patient value - only ciphertexts.')

In [ ]:
colours = {'TenSEAL': C_TS, 'OpenFHE': C_FHE}
stage_labels = ['Encrypt\nfeatures', 'Score\nevaluation', 'Cohort\nmean', 'Decrypt\nscores']

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5))

ax = axes[0]
xs = np.arange(len(STAGES))
n_libs = len(risk_results)
for i, (name, res) in enumerate(risk_results.items()):
    offset = (i - (n_libs - 1) / 2) * (BAR_W + GAP)
    means = [res[f'{s}_time_stats']['mean_ms'] for s in STAGES]
    errs = [res[f'{s}_time_stats']['ci95_half_width_ms'] for s in STAGES]
    bars = ax.bar(xs + offset, means, BAR_W, yerr=errs, capsize=2.5, label=name,
                  color=colours[name], error_kw=dict(ecolor=INK_SOFT, lw=0.8))
    label_bars(ax, bars, means, fmt='{:.0f}')
ax.set_title('Pipeline stage cost'); ax.set_ylabel('Mean time (ms) - log scale')
ax.set_xticks(xs); ax.set_xticklabels(stage_labels, fontsize=8.5)
ax.set_yscale('log'); ax.grid(True, axis='y', alpha=0.6); ax.set_axisbelow(True)
ax.legend(loc='upper right')

ax = axes[1]
ax.hist(reference_scores, bins=40, color=C_PT, edgecolor='white', linewidth=0.5)
mean_score = float(reference_scores.mean())
ax.axvline(mean_score, color=INK, linewidth=1.4)
ax.annotate(f'cohort mean {mean_score:.2f}', xy=(mean_score, ax.get_ylim()[1] * 0.94),
            xytext=(6, 0), textcoords='offset points', fontsize=8.5, color=INK)
ax.set_xlabel('Synthetic risk score (0-100)'); ax.set_ylabel(f'Patients (n = {N_PATIENTS:,})')
ax.set_title('Cohort score distribution')
ax.grid(True, axis='y', alpha=0.6); ax.set_axisbelow(True)

ax = axes[2]
metrics = [('Max abs\nerror', 'score_max_abs_error'),
           ('Mean abs\nerror', 'score_mean_abs_error'),
           ('Cohort mean\nabs error', 'cohort_mean_abs_error')]
xm2 = np.arange(len(metrics))
for i, (name, res) in enumerate(risk_results.items()):
    offset = (i - (n_libs - 1) / 2) * (BAR_W + GAP)
    values = [res[k] for _, k in metrics]
    bars = ax.bar(xm2 + offset, values, BAR_W, label=name, color=colours[name])
    label_bars(ax, bars, values, fmt='{:.1e}')
ax.set_title('Encrypted vs plaintext accuracy')
ax.set_ylabel('Absolute error, score points - log scale')
ax.set_xticks(xm2); ax.set_xticklabels([m[0] for m in metrics], fontsize=8.5)
ax.set_yscale('log'); ax.grid(True, axis='y', alpha=0.6); ax.set_axisbelow(True)
ax.legend(loc='upper right')

plt.suptitle(f'Encrypted synthetic medical risk score - {N_PATIENTS:,} patients, '
             '5 features, interaction and quadratic terms',
             fontsize=13, fontweight='bold')
plt.figtext(0.5, 0.005, 'Synthetic score for demonstration only - not a validated clinical model.',
            ha='center', fontsize=8, color=INK_SOFT)
plt.tight_layout(rect=(0, 0.035, 1, 0.94))
plt.savefig('chart_risk_score.png', dpi=150, bbox_inches='tight')
plt.show()